# Two blocking checks:
**Why:** two assumptions under our regime definitions are not verified, and the paper's main claim rests on them.

1. **Calendar alignment.** Jinay's detector found the weekend dip one day *earlier* than the documented start dates for **both** datasets (offset 6 vs assumed 0 for PEMS04; 3 vs assumed 4 for PEMS08). Either the data really starts a day earlier, or holidays fooled the detector, or the day boundary is not midnight. Cell 3 shows the raw per-day flow so a human can decide; Cell 4 checks that slot 0 really is midnight.
2. **Is the regime-graph difference real or just noise?** The peak graph uses fewer slots and the weekend graph about 10 days, so graphs could differ from the global one just by sampling noise. Cell 5 builds each regime graph from two random halves of the days: if two halves of the *same* regime agree much more than different regimes do, the difference is real.

**Settings:** Internet On (installs POT), no GPU needed. Add Data: PEMS04 and PEMS08 (`.npz`). **Send the printed output of Cells 3-5 and `alignment_checks.png` to the group.**

In [ ]:
import os, sys, glob, subprocess, time, json
from pathlib import Path
import numpy as np, pandas as pd
try: import ot
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pot'], check=True); import ot
ON_KAGGLE = os.path.exists('/kaggle'); WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./working'); WORK.mkdir(exist_ok=True, parents=True)
STAD_LIB = r'''import numpy as np, ot, warnings
from multiprocessing import Pool
warnings.filterwarnings('ignore')

def prep_node_vectors(data, eps=1e-8):
    # data: (days, L, N) -> unit vectors Xn (N, days, L) and masses P (N, days)
    X = np.ascontiguousarray(np.transpose(data, (2, 0, 1)).astype(np.float64))
    nrm = np.linalg.norm(X, axis=2)                                   # (N, days) daily volume (Eq. 3)
    Xn = X / np.maximum(nrm, eps)[:, :, None]
    s = nrm.sum(1, keepdims=True)
    P = np.where(s > 0, nrm / np.maximum(s, eps), 1.0 / X.shape[1])
    return np.ascontiguousarray(Xn), np.ascontiguousarray(P)

_G = {}
def _row(i):
    Xn, P = _G['Xn'], _G['P']; N = Xn.shape[0]; out = np.zeros(N)
    for j in range(i + 1, N):
        C = np.ascontiguousarray(1.0 - Xn[i] @ Xn[j].T)                                   # cosine cost (Eq. 4)
        out[j] = 1.0 - ot.emd2(P[i], P[j], C)                          # similarity = 1 - Wasserstein (Eq. 5)
    return i, out

def stad_matrix(data, n_proc=1):
    # returns symmetric (N,N) similarity A_STAD with zero diagonal
    Xn, P = prep_node_vectors(data); _G['Xn'], _G['P'] = Xn, P; N = Xn.shape[0]
    d = np.zeros((N, N))
    if n_proc > 1:
        with Pool(n_proc) as pool:
            for i, r in pool.imap_unordered(_row, range(N), chunksize=2): d[i] = r
    else:
        for i in range(N): d[i] = _row(i)[1]
    return np.nan_to_num(d + d.T)

def make_graphs(sta, sparsity=0.01, stag_include_self=True):
    # Returns (A_STAG, A_STRG).
    # A_STRG (attention prior): authors' released STAG_gen.py logic = top-k OTHER nodes, plus the diagonal (weighted).
    # A_STAG (Chebyshev graph): the authors' SHIPPED PEMS04 file is "self + (k-1) others" (verified: 91% of rows identical when
    #   the diagonal similarity is set to 1), so with stag_include_self=True we reproduce that convention. Self-loops cancel in L = D - A,
    #   so this means k-1 real neighbours per node in the graph convolution.
    N = sta.shape[0]; I = np.identity(N)
    adjl = sta + I
    adjlnormd = adjl / adjl.mean(axis=0)
    adj = 1 - adjl + I
    A = np.zeros((N, N)); R = np.zeros((N, N)); top = int(N * sparsity)
    s2 = sta.copy(); np.fill_diagonal(s2, 1.0)
    for i in range(N):
        a = adj[i, :].argsort()[0:top]
        R[i, a] = adjlnormd[i, a]
        R[i, i] = adjlnormd[i, i]
        if stag_include_self:
            A[i, np.argsort(-s2[i], kind='stable')[:top]] = 1
        else:
            A[i, a] = 1
    return A, R
'''
(WORK / 'stad_lib.py').write_text(STAD_LIB); sys.path.insert(0, str(WORK)); from stad_lib import stad_matrix
DOCUMENTED_START = {'PEMS04': ('2018-01-01', 'Mon'), 'PEMS08': ('2016-07-01', 'Fri')}   # from the dataset descriptions
DATASETS = ['PEMS04', 'PEMS08']; RAW = {}
for ds in DATASETS:
    pat = '/kaggle/input/**/%s.npz' % ds if ON_KAGGLE else './**/%s.npz' % ds
    h = glob.glob(pat, recursive=True); assert h, 'Add the %s dataset (needs %s.npz)' % (ds, ds)
    RAW[ds] = np.load(h[0])['data'][:, :, 0]; print(ds, RAW[ds].shape)
N_CPU = os.cpu_count() or 1

## Cell 3: Flow per day, first 3 weeks, labelled with the DOCUMENTED calendar
Look at the `bar` column. Expected: five similar weekdays, then two clearly lower days (Sat, Sun). Check which calendar labels the two low days carry, and whether a low day coincides with a holiday (PEMS04: Mon 1 Jan, Mon 15 Jan; PEMS08: Mon 4 Jul).

In [ ]:
for ds in DATASETS:
    T, N = RAW[ds].shape; nd = T // 288
    daily = RAW[ds][:nd * 288].reshape(nd, 288, N).mean(axis=(1, 2)); start = pd.Timestamp(DOCUMENTED_START[ds][0])
    print('\n%s: documented start %s (%s)' % (ds, DOCUMENTED_START[ds][0], DOCUMENTED_START[ds][1]))
    top = daily[:21].max()
    for d in range(21):
        dt = start + pd.Timedelta(days=d); print('  day %2d  %s %s  flow %6.1f  %s' % (d, dt.date(), dt.strftime('%a'), daily[d], '#' * int(40 * daily[d] / top)))
    med = {k: float(np.median([daily[d] for d in range(nd) if (start + pd.Timedelta(days=d)).strftime('%a') == k])) for k in ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']}
    print('  MEDIAN flow by documented weekday:', {k: round(v, 1) for k, v in med.items()})
    low2 = sorted(med, key=med.get)[:2]; print('  two lowest documented weekdays:', low2, '| expected [Sat, Sun] -> ', 'CONSISTENT' if set(low2) == {'Sat', 'Sun'} else '*** NOT consistent with the documented start date ***')

## Cell 4: Is slot 0 midnight? (average daily profile)
Typical California freeway flow: lowest around 02:00-04:30, morning peak 07:00-09:30, evening peak 15:30-18:00. If the night trough is near slot 24-54 and peaks near slots 84-114 and 186-216, then slot index = clock time x 12 and our peak windows (slots 84-120 and 192-228) are right. A shift of 60-100 slots would mean the day boundary is not at midnight.

In [ ]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
for a, ds in zip(ax, DATASETS):
    T, N = RAW[ds].shape; prof = RAW[ds][:(T // 288) * 288].reshape(-1, 288, N).mean(axis=(0, 2))
    sm = np.convolve(np.r_[prof[-6:], prof, prof[:6]], np.ones(13) / 13, mode='valid')
    trough = int(np.argmin(sm)); am = 48 + int(np.argmax(sm[48:150])); pm = 150 + int(np.argmax(sm[150:270]))
    clock = lambda s: '%02d:%02d' % (s * 5 // 60, s * 5 % 60)
    print('%s: night trough slot %d (%s) | morning peak slot %d (%s) | evening peak slot %d (%s) | peak windows used: 84-120 (07-10h), 192-228 (16-19h)' % (ds, trough, clock(trough), am, clock(am), pm, clock(pm)))
    a.plot(np.arange(288) / 12, prof); a.axvspan(7, 10, alpha=.15); a.axvspan(16, 19, alpha=.15); a.set_title(ds + ' mean flow vs slot/12 (shaded = our peak windows)'); a.set_xlabel('slot / 12')
plt.tight_layout(); plt.savefig(WORK / 'alignment_checks.png', dpi=140); plt.show()

## Cell 5: Noise control: are regime graphs more different from each other than two halves of the *same* regime?
Uses the weekday definition below (`START_DOW`). **Set it from what Cell 3 showed** (0 = Monday; the day-of-week of day 0). Overlap = mean Jaccard of each node's top-k neighbour set at k = 5% of nodes.

In [ ]:
START_DOW = {'PEMS04': 0, 'PEMS08': 4}      # documented start dates (PEMS04 = 0 is CONFIRMED by the data; PEMS08 = 4 is consistent with it, see Cell 3). Do NOT use 6 / 3: that was the automatic detector, fooled by low Mondays
REPEATS = 2; K_FRAC = 0.05; rng = np.random.default_rng(0)
PEAK = np.r_[np.arange(84, 120), np.arange(192, 228)]; OFF = np.setdiff1d(np.arange(288), PEAK)
def nbr_sets(S, k):
    s = S.copy(); np.fill_diagonal(s, -1); return [set(np.argsort(-s[i])[:k]) for i in range(len(s))]
def jac(a, b): return float(np.mean([len(x & y) / max(len(x | y), 1) for x, y in zip(a, b)]))
out = []
for ds in DATASETS:
    T, N = RAW[ds].shape; nd = int(int(T * 0.6) / 288); data = RAW[ds][:nd * 288].reshape(nd, 288, N)
    dow = (np.arange(nd) + START_DOW[ds]) % 7; wk, we = np.where(dow < 5)[0], np.where(dow >= 5)[0]; k = max(int(N * K_FRAC), 1)
    regs = {'weekday_peak': (wk, PEAK), 'weekday_offpeak': (wk, OFF), 'weekend': (we, np.arange(288))}
    full = {n: nbr_sets(stad_matrix(data[np.ix_(d, s)], N_CPU), k) for n, (d, s) in regs.items()}
    for n, (d, s) in regs.items():
        same = []
        for _ in range(REPEATS):
            p = rng.permutation(d); h1, h2 = p[:len(p) // 2], p[len(p) // 2:]
            same.append(jac(nbr_sets(stad_matrix(data[np.ix_(h1, s)], N_CPU), k), nbr_sets(stad_matrix(data[np.ix_(h2, s)], N_CPU), k)))
        out.append(dict(dataset=ds, regime=n, days=len(d), half_vs_half_same_regime=np.mean(same)))
        print(ds, n, 'days', len(d), 'split-half overlap %.3f' % np.mean(same), flush=True)
    names = list(regs)
    for i, a in enumerate(names):
        for b in names[i + 1:]: out.append(dict(dataset=ds, regime=a + ' vs ' + b, cross_regime=jac(full[a], full[b])))
res = pd.DataFrame(out); print('\n', res.round(3).to_string(index=False)); res.to_csv(WORK / 'noise_control.csv', index=False)
print('''\nREAD IT LIKE THIS: if cross-regime overlap (e.g. weekday_peak vs weekday_offpeak) is clearly BELOW the split-half overlap of each regime, the regime graphs differ by more than sampling noise
(good for the paper). If they are about equal, the differences are mostly noise: tell the group before Advik's results are interpreted.''')